# `alpaca_executor.py` — Playground

Manual verification notebook for the **Alpaca execution layer**: live account state (portfolio value, open positions, daily P&L, drawdown) and order placement.

| Function | Status | Notes |
|---|---|---|
| `get_portfolio_value()` | ✅ built | Total account equity (cash + positions) |
| `get_open_positions()` | ✅ built | Open positions with qty / market value / unrealized P&L |
| `get_daily_pnl()` | ✅ built | Today's P&L (equity − last_equity) |
| `get_drawdown_pct()` | ✅ built | Drawdown from peak equity over the lookback period |
| `position_trade()` | ✅ built | Entry + trailing stop; **requires market open** (no after-hours queue) |

**Exit design (Phase 6).** Alpaca supports a `trailing_stop` only as a *single* order — it cannot be the stop leg of a bracket — so a trade is **two orders**: a market entry, then a standalone trailing stop once the entry fills. Alpaca ratchets the stop against the high-water mark broker-side, so there is no monitoring loop.

The trail is **derived**, not a config dial: `trail_percent = trade_levels['stop_pct'] × 100`. That is the same ATR stop distance Gate 5's EV and the risk gate's Kelly sizing were computed from, so the order placed in the market describes the trade that was actually authorised.


In [11]:
import sys
import pathlib

exec_dir = pathlib.Path('.').resolve()
if not (exec_dir / 'alpaca_executor.py').exists():
    exec_dir = pathlib.Path('backend/04_execution').resolve()

if str(exec_dir) not in sys.path:
    sys.path.insert(0, str(exec_dir))

from alpaca_executor import (
    get_portfolio_value,
    get_open_positions,
    get_daily_pnl,
    get_drawdown_pct,
    position_trade,
    _get_alpaca_client,
)


---
## Happy path — live paper account state

Requires `ALPACA_API_KEY` / `ALPACA_SECRET_KEY` / `ALPACA_BASE_URL` in `.env`.


In [2]:
get_portfolio_value()

100000.58

In [3]:
get_open_positions()

[]

In [4]:
get_daily_pnl()

0.5800000000017462

In [5]:
get_drawdown_pct()

0.0

---
## Happy path — position a live paper trade

Places a **real paper order**: 1 share of AAPL plus a trailing stop, then cleans up after itself.

`position_trade()` only submits when the market is open. Outside RTH it returns `None`
immediately — otherwise Alpaca would queue the order as `accepted` with filled qty 0.


In [12]:
api = _get_alpaca_client()

audit = position_trade({
    'ticker': 'AAPL',
    'shares': 1,
    'trade_levels': {'stop_pct': 0.0211},   # the NVDA reference candidate -> a 2.11% trail
})

if audit:
    audit
else:
    print('No position opened (market closed, or entry failed).')
    if api is not None:
        print('Next open:', api.get_clock().next_open)


[executor] AAPL: entry did not fill within 60s — cancelling
No position opened (market closed, or entry failed).
Next open: 2026-07-14 09:30:00-04:00


In [9]:
# What Alpaca is now holding for us, broker-side
if audit and audit['stop_attached']:
    for o in api.list_orders(status='open'):
        print(f'{o.symbol}  {o.type}  {o.side}  qty={o.qty}  trail={o.trail_percent}%  '
              f'hwm={o.hwm}  stop={o.stop_price}')
    print()
    print('stop_price should equal hwm x (1 - trail/100):',
          round(float(audit['hwm']) * (1 - audit['trail_percent'] / 100), 2))


In [8]:
# Clean up — cancel the trailing stop and close the test position
if audit:
    api.cancel_all_orders()
    api.close_position(audit['ticker'])
    print('cleaned up', audit['ticker'])


---
## Free-play

Scratch cell — re-run pieces above. Live fills only work during regular market hours (ET).
